# 제18장: 공공 AI 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch18.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

공공 AI는 민주적 책임성과 시민 권익 보호가 핵심 거버넌스 가치입니다.
**알고리즘 영향 평가(AIA), IT 서비스 관리 거버넌스, 인시던트 대응,
공공 AI 투명성 보고서** 자동 생성 도구를 구현합니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**공공 AI Equity Assessment and Vulnerable Group Impact Analysis Framework**

In [ ]:
from dataclasses import dataclass
from enum import Enum
import pandas as pd
import numpy as np

class VulnerableGroup(Enum):
 ELDERLY = " "; DISABLED = " "; LOW_INCOME = " "
 RURAL = " "; DIGITAL_VULNERABLE = " "

class PublicAIEquityEvaluator:
 """ Equity Assessment : Metric """
 def __init__(self, disparity_threshold=0.15, critical_threshold=0.30):
 self.disparity_threshold = disparity_threshold
 self.critical_threshold = critical_threshold

 def evaluate_equity(self, data, group_col, outcome_col, groups):
 results = []
 general_mask = ~data[group_col].isin([g.value for g in groups])
 general_val = data[general_mask][outcome_col].mean()

 for group in groups:
 group_val = data[data[group_col] == group.value][outcome_col].mean()
 disparity = (general_val - group_val) / general_val if general_val != 0 else 0

 status = " "
 if abs(disparity) > self.critical_threshold: status = " "
 elif abs(disparity) > self.disparity_threshold: status = " "

 results.append({"group": group.value, "disparity": disparity, "status": status})
 return results

**Administrative Basic Act Article 20 Compliance Checker for Public AI**

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from enum import Enum
from datetime import datetime

class DispositionType(Enum):
    BOUND = "bound_act"        # (fully automatable)
    LIMITED_DISCRETION = "limited_discretion"  # (AI-assisted)
    BROAD_DISCRETION = "broad_discretion"      # (no automation)

class AutomationLevel(Enum):
    FULL_AUTO = "full_automation"
    SEMI_AUTO = "semi_automation"
    AI_ASSISTED = "ai_assisted"
    HUMAN_ONLY = "human_only"

@dataclass
class DispositionAssessment:
    disposition_name: str
    disposition_type: DispositionType
    legal_basis: str
    allowed_automation: AutomationLevel
    requires_explanation: bool
    requires_appeal_notice: bool
    human_review_required: bool
    compliance_issues: List[str]

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class Article20ComplianceChecker:
    """Check AI system compliance with Administrative Basic Act Art.20"""

    AUTOMATION_RULES = {
        DispositionType.BOUND: AutomationLevel.FULL_AUTO,
        DispositionType.LIMITED_DISCRETION: AutomationLevel.SEMI_AUTO,
        DispositionType.BROAD_DISCRETION: AutomationLevel.HUMAN_ONLY,
    }

    def __init__(self):
        self.assessments: List[DispositionAssessment] = []

    def assess_disposition(self, name: str, dtype: DispositionType,
                           legal_basis: str,
                           current_automation: AutomationLevel) -> DispositionAssessment:
        """Assess whether current automation level is legally compliant"""
        allowed = self.AUTOMATION_RULES[dtype]
        issues = []

        # Check automation level compliance
        auto_order = [AutomationLevel.HUMAN_ONLY, AutomationLevel.AI_ASSISTED,
                     AutomationLevel.SEMI_AUTO, AutomationLevel.FULL_AUTO]
        if auto_order.index(current_automation) > auto_order.index(allowed):
            issues.append(
                f"Automation level '{current_automation.value}' exceeds "
                f"allowed level '{allowed.value}' for {dtype.value}"
            )

        # Legal basis check
        if not legal_basis or legal_basis.strip() == "":
            issues.append("Missing legal basis for automated disposition")

        # Discretion check
        human_review = dtype != DispositionType.BOUND

        assessment = DispositionAssessment(
            disposition_name=name,
            disposition_type=dtype,
            legal_basis=legal_basis,
            allowed_automation=allowed,
            requires_explanation=True,  # Always required for public AI
            requires_appeal_notice=True,  # Always required
            human_review_required=human_review,
            compliance_issues=issues
        )
        self.assessments.append(assessment)
        return assessment

    def generate_compliance_report(self) -> Dict:
        """Generate Article 20 compliance report"""
        total = len(self.assessments)
        compliant = sum(1 for a in self.assessments if not a.compliance_issues)

        return {
            "report_date": datetime.now().isoformat(),
            "total_dispositions": total,
            "compliant": compliant,
            "non_compliant": total - compliant,
            "issues": [
                {"disposition": a.disposition_name, "issues": a.compliance_issues}
                for a in self.assessments if a.compliance_issues
            ],
            "overall_status": "COMPLIANT" if compliant == total else "ACTION_REQUIRED"
        }

**Public AI Impact Assessment Automation Tool**

In [ ]:
import json
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from enum import Enum

class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    VERY_HIGH = "very_high"

class ImpactCategory(Enum):
    FUNDAMENTAL_RIGHTS = "fundamental_rights"
    EQUITY = "equity"
    PRIVACY = "privacy"
    TRANSPARENCY = "transparency"
    SAFETY = "safety"
    ACCOUNTABILITY = "accountability"

@dataclass
class ImpactAssessmentItem:
    category: ImpactCategory
    question: str
    score: int  # 1-5
    weight: float
    evidence: str
    mitigation: str

@dataclass
class PublicAIImpactReport:
    system_name: str
    agency: str
    assessment_date: datetime
    items: List[ImpactAssessmentItem]
    overall_risk: RiskLevel
    recommendations: List[str]
    approval_status: str

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class PublicAIImpactAssessor:
    """Automated impact assessment tool for public AI systems"""

    ASSESSMENT_TEMPLATE = {
        ImpactCategory.FUNDAMENTAL_RIGHTS: [
            "AI system may restrict citizen's right to due process",
            "AI system may affect freedom of expression or assembly",
            "AI decisions may be difficult for citizens to challenge",
            "Vulnerable groups may be disproportionately affected",
        ],
        ImpactCategory.EQUITY: [
            "AI system produces different outcomes across regions",
            "Elderly or disabled citizens have reduced access",
            "Low-income groups face higher error rates",
            "Digital literacy gaps affect service quality",
        ],
        ImpactCategory.PRIVACY: [
            "Personal data is collected beyond minimum necessary",
            "Data retention exceeds required period",
            "Cross-agency data sharing lacks proper controls",
            "Re-identification risk from combined datasets",
        ],
        ImpactCategory.TRANSPARENCY: [
            "AI decision logic is not explainable to citizens",
            "System is not registered in algorithm register",
            "Performance metrics are not publicly available",
            "Citizens cannot request explanation of decisions",
        ],
        ImpactCategory.SAFETY: [
            "System failure could cause significant citizen harm",
            "No fallback procedure exists for system outage",
            "Adversarial attacks could manipulate outcomes",
            "No regular security assessment is performed",
        ],
        ImpactCategory.ACCOUNTABILITY: [
            "Responsibility chain for AI errors is unclear",
            "No human oversight mechanism exists",
            "Appeal/grievance process is not established",
            "No regular audit cycle is defined",
        ],
    }

    CATEGORY_WEIGHTS = {
        ImpactCategory.FUNDAMENTAL_RIGHTS: 0.25,
        ImpactCategory.EQUITY: 0.20,
        ImpactCategory.PRIVACY: 0.15,
        ImpactCategory.TRANSPARENCY: 0.15,
        ImpactCategory.SAFETY: 0.15,
        ImpactCategory.ACCOUNTABILITY: 0.10,
    }

    def __init__(self):
        self.assessments: List[PublicAIImpactReport] = []

    def conduct_assessment(self, system_name: str, agency: str,
                           responses: Dict[str, List[Dict]]) -> PublicAIImpactReport:
        """Conduct automated impact assessment"""
        items = []

        for category, questions in self.ASSESSMENT_TEMPLATE.items():
            cat_responses = responses.get(category.value, [])
            weight = self.CATEGORY_WEIGHTS[category]

            for i, question in enumerate(questions):
                resp = cat_responses[i] if i < len(cat_responses) else {}
                items.append(ImpactAssessmentItem(
                    category=category,
                    question=question,
                    score=resp.get("score", 3),
                    weight=weight / len(questions),
                    evidence=resp.get("evidence", ""),
                    mitigation=resp.get("mitigation", "")
                ))

        # Calculate overall risk
        weighted_score = sum(item.score * item.weight for item in items)
        if weighted_score >= 4.0:
            overall_risk = RiskLevel.VERY_HIGH
        elif weighted_score >= 3.0:
            overall_risk = RiskLevel.HIGH
        elif weighted_score >= 2.0:
            overall_risk = RiskLevel.MEDIUM
        else:
            overall_risk = RiskLevel.LOW

        # Generate recommendations
        recommendations = self._generate_recommendations(items, overall_risk)

        # Determine approval status
        if overall_risk == RiskLevel.VERY_HIGH:
            approval = "REJECTED - Major redesign required"
        elif overall_risk == RiskLevel.HIGH:
            approval = "CONDITIONAL - Mitigation measures required before deployment"
        elif overall_risk == RiskLevel.MEDIUM:
            approval = "APPROVED WITH CONDITIONS - Enhanced monitoring required"
        else:
            approval = "APPROVED - Standard monitoring"

        report = PublicAIImpactReport(
            system_name=system_name,
            agency=agency,
            assessment_date=datetime.now(),
            items=items,
            overall_risk=overall_risk,
            recommendations=recommendations,
            approval_status=approval
        )

        self.assessments.append(report)
        return report

    def _generate_recommendations(self, items: List[ImpactAssessmentItem],
                                   risk: RiskLevel) -> List[str]:
        """Generate context-specific recommendations"""
        recs = []
        high_risk_items = [i for i in items if i.score >= 4]

        for item in high_risk_items:
            if item.category == ImpactCategory.FUNDAMENTAL_RIGHTS:
                recs.append("Implement mandatory human review for high-impact decisions")
            elif item.category == ImpactCategory.EQUITY:
                recs.append("Conduct quarterly equity audit with disaggregated data")
            elif item.category == ImpactCategory.PRIVACY:
                recs.append("Strengthen data minimization and retention policies")
            elif item.category == ImpactCategory.TRANSPARENCY:
                recs.append("Register system in algorithm transparency register")
            elif item.category == ImpactCategory.SAFETY:
                recs.append("Develop and test fallback procedures")
            elif item.category == ImpactCategory.ACCOUNTABILITY:
                recs.append("Establish clear accountability chain and audit schedule")

        if risk in [RiskLevel.HIGH, RiskLevel.VERY_HIGH]:
            recs.append("Conduct public consultation before deployment")
            recs.append("Establish citizen audit panel for ongoing monitoring")

        return recs

    def export_report(self, report: PublicAIImpactReport) -> Dict:
        """Export assessment report in structured format"""
        return {
            "system_name": report.system_name,
            "agency": report.agency,
            "assessment_date": report.assessment_date.isoformat(),
            "overall_risk": report.overall_risk.value,
            "approval_status": report.approval_status,
            "category_scores": {
                cat.value: np.mean([i.score for i in report.items if i.category == cat])
                for cat in ImpactCategory
            },
            "recommendations": report.recommendations,
            "total_items_assessed": len(report.items),
            "high_risk_items": sum(1 for i in report.items if i.score >= 4)
        }

**Public Chatbot Governance: Answer Quality and Escalation Management**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime, timedelta
from enum import Enum
from collections import defaultdict

class QueryCategory(Enum):
    FACTUAL_INFO = "factual_info"        # (automatable)
    PROCEDURE_GUIDE = "procedure_guide"  # (automatable)
    LEGAL_INTERPRETATION = "legal"       # (escalation required)
    COMPLAINT = "complaint"              # (escalation required)
    COMPLEX_CASE = "complex"             # (escalation required)

class AnswerStatus(Enum):
    ACTIVE = "active"
    UNDER_REVIEW = "under_review"
    DEACTIVATED = "deactivated"
    UPDATED = "updated"

@dataclass
class ChatbotAnswer:
    answer_id: str
    question_pattern: str
    answer_text: str
    category: QueryCategory
    status: AnswerStatus
    legal_reference: str
    last_verified: datetime
    negative_feedback_count: int = 0
    total_served: int = 0

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class PublicChatbotGovernance:
    """Governance system for public service chatbot"""

    ESCALATION_CATEGORIES = [
        QueryCategory.LEGAL_INTERPRETATION,
        QueryCategory.COMPLAINT,
        QueryCategory.COMPLEX_CASE
    ]

    def __init__(self, negative_feedback_threshold: int = 3,
                 answer_refresh_days: int = 30):
        self.answers: Dict[str, ChatbotAnswer] = {}
        self.neg_threshold = negative_feedback_threshold
        self.refresh_days = answer_refresh_days
        self.escalation_log: List[Dict] = []

    def process_query(self, query_text: str, category: QueryCategory,
                      matched_answer_id: Optional[str]) -> Dict:
        """Process citizen query with governance checks"""

        # Check if escalation is required
        if category in self.ESCALATION_CATEGORIES:
            self._log_escalation(query_text, category, "Category requires human review")
            return {
                "action": "escalate",
                "message": "This question requires expert review. "
                          "Transferring to a staff member.",
                "estimated_response": "Within 2 business hours"
            }

        if not matched_answer_id or matched_answer_id not in self.answers:
            self._log_escalation(query_text, category, "No matching answer found")
            return {
                "action": "escalate",
                "message": "Unable to find a matching answer. "
                          "Transferring to a staff member."
            }

        answer = self.answers[matched_answer_id]

        # Check answer status
        if answer.status != AnswerStatus.ACTIVE:
            self._log_escalation(query_text, category,
                               f"Answer status: {answer.status.value}")
            return {"action": "escalate", "message": "Answer is under review."}

        # Check answer freshness
        if (datetime.now() - answer.last_verified).days > self.refresh_days:
            answer.status = AnswerStatus.UNDER_REVIEW
            return {"action": "escalate",
                    "message": "Answer requires verification update."}

        answer.total_served += 1
        return {
            "action": "respond",
            "answer": answer.answer_text,
            "legal_reference": answer.legal_reference,
            "feedback_prompt": "Was this answer helpful? [Yes/No]"
        }

    def record_feedback(self, answer_id: str, is_positive: bool):
        """Record citizen feedback on answer quality"""
        if answer_id not in self.answers:
            return

        answer = self.answers[answer_id]
        if not is_positive:
            answer.negative_feedback_count += 1

            if answer.negative_feedback_count >= self.neg_threshold:
                answer.status = AnswerStatus.DEACTIVATED
                self._log_escalation(
                    f"Auto-deactivated: {answer.question_pattern}",
                    QueryCategory.FACTUAL_INFO,
                    f"Negative feedback count: {answer.negative_feedback_count}"
                )

    def _log_escalation(self, query: str, category: QueryCategory, reason: str):
        """Log escalation for audit trail"""
        self.escalation_log.append({
            "timestamp": datetime.now().isoformat(),
            "query": query[:200],
            "category": category.value,
            "reason": reason
        })

    def generate_quality_report(self) -> Dict:
        """Generate monthly quality report"""
        total_answers = len(self.answers)
        active = sum(1 for a in self.answers.values()
                     if a.status == AnswerStatus.ACTIVE)
        deactivated = sum(1 for a in self.answers.values()
                         if a.status == AnswerStatus.DEACTIVATED)
        total_escalations = len(self.escalation_log)

        return {
            "report_period": datetime.now().strftime("%Y-%m"),
            "total_answers_in_db": total_answers,
            "active_answers": active,
            "deactivated_answers": deactivated,
            "total_escalations": total_escalations,
            "answer_health_rate": active / total_answers if total_answers > 0 else 0
        }

**ITS AI Surveillance and Safety Governance Management System**

In [ ]:
from enum import Enum

class IncidentType(Enum):
    """공공 교통 AI 시스템 인시던트 유형. AI 기본법상 고영향 AI 사고 보고 의무 대상."""
    CONGESTION = "congestion"          # 교통 정체 — 자동 처리 가능
    ACCIDENT = "accident"              # 교통 사고 — 인간 검토 필수
    WRONG_WAY = "wrong_way"            # 역주행 — 즉시 경보
    ROAD_CLOSURE = "road_closure"      # 도로 폐쇄
    WEATHER = "weather"                # 기상 악화

class ITSGovernanceManager:
 """Proper Role Allocation between ITS AI Recommendations and Human Control"""
 def __init__(self):
 self.auto_execute_types = [IncidentType.CONGESTION] # self.critical_types = [IncidentType.ACCIDENT, IncidentType.WRONG_WAY]

 def process_recommendation(self, rec):
 # (Safety-First)
 if rec.incident_type in self.critical_types:
 return {"action": "human_review_required", "urgency": "EMERGENCY"}

 # 95% days (Efficiency)
 if rec.incident_type in self.auto_execute_types and rec.confidence > 0.95:
 return {"action": "auto_execute", "logging": True}

 return {"action": "human_review_required", "urgency": "NORMAL"}

**Public AI Governance Maturity Self-Assessment Tool**

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
from enum import Enum

class MaturityLevel(Enum):
    INITIAL = 1
    AWARE = 2
    SYSTEMATIC = 3
    MANAGED = 4
    OPTIMIZED = 5

class AssessmentDomain(Enum):
    POLICY = "policy"
    ORGANIZATION = "organization"
    PROCESS = "process"
    TECHNOLOGY = "technology"
    CITIZEN_PARTICIPATION = "citizen_participation"
    DATA_GOVERNANCE = "data_governance"
    RISK_MANAGEMENT = "risk_management"

@dataclass
class DomainAssessment:
    domain: AssessmentDomain
    score: int  # 1-5
    evidence: str
    improvement_actions: List[str]

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class PublicAIMaturityAssessor:
    """Self-assessment tool for public AI governance maturity"""

    DOMAIN_CRITERIA = {
        AssessmentDomain.POLICY: {
            1: "No AI governance policy exists",
            2: "Basic AI ethics principles declared",
            3: "Formal governance regulations established",
            4: "Detailed guidelines per AI system type",
            5: "Self-regulating adaptive governance framework"
        },
        AssessmentDomain.ORGANIZATION: {
            1: "No designated AI governance personnel",
            2: "Part-time AI governance coordinator assigned",
            3: "Dedicated AI governance team established",
            4: "CDO/CAIO role with authority and budget",
            5: "Cross-agency governance coordination body"
        },
        AssessmentDomain.PROCESS: {
            1: "Ad-hoc AI deployment without review",
            2: "Basic review before AI deployment",
            3: "Standardized impact assessment process",
            4: "Automated compliance checking integrated",
            5: "Continuous improvement with feedback loops"
        },
        AssessmentDomain.TECHNOLOGY: {
            1: "No monitoring or XAI tools",
            2: "Basic performance logging",
            3: "XAI and bias monitoring deployed",
            4: "Real-time governance dashboard",
            5: "AI self-diagnosis and auto-remediation"
        },
        AssessmentDomain.CITIZEN_PARTICIPATION: {
            1: "No citizen engagement mechanism",
            2: "Information disclosure only",
            3: "Public hearings and comment periods",
            4: "Active citizen audit panels",
            5: "Co-design and participatory governance"
        },
        AssessmentDomain.DATA_GOVERNANCE: {
            1: "No data governance for AI",
            2: "Basic data quality checks",
            3: "Pseudonymization and access controls",
            4: "Data fabric with automated compliance",
            5: "Privacy-by-design with citizen data sovereignty"
        },
        AssessmentDomain.RISK_MANAGEMENT: {
            1: "No AI risk assessment",
            2: "Initial risk identification",
            3: "Structured risk assessment framework",
            4: "Quantitative risk monitoring with KPIs",
            5: "Predictive risk management with early warning"
        }
    }

    def __init__(self):
        self.assessments: List[DomainAssessment] = []

    def assess_domain(self, domain: AssessmentDomain,
                      score: int, evidence: str) -> DomainAssessment:
        """Assess a single governance domain"""
        improvements = []
        if score < 5:
            next_level = self.DOMAIN_CRITERIA[domain].get(score + 1, "")
            improvements.append(f"Target: {next_level}")

        assessment = DomainAssessment(
            domain=domain, score=score,
            evidence=evidence, improvement_actions=improvements
        )
        self.assessments.append(assessment)
        return assessment

    def calculate_overall_maturity(self) -> Dict:
        """Calculate overall maturity level"""
        if not self.assessments:
            return {"level": MaturityLevel.INITIAL, "score": 1.0}

        avg_score = sum(a.score for a in self.assessments) / len(self.assessments)

        # Overall level is constrained by the lowest domain
        min_score = min(a.score for a in self.assessments)
        effective_level = min(int(avg_score), min_score + 1)

        level = MaturityLevel(min(effective_level, 5))

        return {
            "level": level,
            "average_score": round(avg_score, 2),
            "min_domain_score": min_score,
            "weakest_domain": min(self.assessments, key=lambda a: a.score).domain.value,
            "domain_scores": {a.domain.value: a.score for a in self.assessments},
            "recommendation": f"Focus on improving '{min(self.assessments, key=lambda a: a.score).domain.value}' "
                            f"to advance overall maturity"
        }